# Full evaluation of the winner

Runtime: **L4 GPU**, its own runtime. Mount Google Drive first (Files panel, Drive icon): every result and adapter is copied to MyDrive/thinkingcap-mvtf and restored from there. Then Run all. The last cell works through `plan_rebuild.txt` from the branch and keeps polling it, so new stages are queued by pushing a line.

In [ ]:
# Google Drive first: every model and score is restored from it and saved to it
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%env PYTHONUNBUFFERED=1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d ThinkingCap-Test || git clone -q https://github.com/BottleCapAI/ThinkingCap-Test
%cd /content/ThinkingCap-Test
!test -f .installed || (pip -q install uv && uv pip install --system -q -r requirements.txt && touch .installed)

In [ ]:
!rm -rf /content/T && git clone -q --depth 1 -b mvt-fading https://github.com/arsenelemoigne/Test /content/T && cp /content/T/thinkingcap/*.py /content/T/thinkingcap/plan*.txt . && git -C /content/T log -1 --format='%h %s'

In [ ]:
import os
try:  # the OpenRouter key for the astra stage, from Colab's Secrets panel (never printed, never in a file)
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    print('OpenRouter key loaded from Colab Secrets (not shown)')
except Exception as e:
    print('no OpenRouter key from Colab Secrets:', type(e).__name__)
try:  # the Delos worker key (astra --as-provider delos), same rules
    from google.colab import userdata
    os.environ['DELOS_API_KEY'] = userdata.get('DELOS_API_KEY')
    print('Delos key loaded from Colab Secrets (not shown)')
except Exception as e:
    print('no Delos key from Colab Secrets:', type(e).__name__)
try:  # the user's Google AI Studio key (astra --as-provider google: Gemini 4 Argon), same rules
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('Google key loaded from Colab Secrets (not shown)')
except Exception as e:
    print('no Google key from Colab Secrets:', type(e).__name__)
!python track.py setup

In [ ]:
!python track.py plan --plan-file plan_line_b.txt --wait-hours 0.2 2>&1 | tee -a /content/track_log.txt

In [ ]:
# queue finished: flush Drive (in the kernel, where it works), then release this runtime
import time
from google.colab import drive, runtime
try:
    drive.flush_and_unmount()
except Exception as error:
    print(error)
time.sleep(60)
runtime.unassign()